# Spark Foundations & Execution Model

## What's covered

- What Spark is and where it earns its complexity over pandas
- Cluster anatomy: driver, executors, cluster manager
- How a chain of transformations becomes jobs, stages, and tasks
- Deployment modes — local, standalone, YARN, Kubernetes
- When NOT to use Spark
- SparkSession setup — the canonical pattern reused throughout this series
- Lazy evaluation — the first principle that makes everything else possible
- Catalyst — the four-phase rule-based optimizer
- Tungsten — off-heap memory and whole-stage codegen
- Reading `.explain()` plans
- Adaptive Query Execution (AQE)
- Spark Connect — the split-driver architecture and when to use it


## The problem Spark solves

Imagine a CSV with two hundred gigabytes of bank transactions and a laptop with sixteen gigabytes of RAM. Pandas crashes the moment you try to load it. You could split the file into chunks and loop, but you now own a small distributed system: which chunks finished, which failed, how do you do a group-by that spans the whole file?

Spark exists because this pattern — data too big for one machine, work that needs to be coordinated across many — keeps appearing past a few hundred gigabytes. Instead of writing your own chunking and coordination layer, Spark gives you one engine that handles it, behind a familiar table-shaped API.


## What Spark actually is

Apache Spark is a **distributed compute engine**. You write code as if you have one giant table. Spark slices the data into pieces called **partitions**, assigns each piece to a machine — telling it which file and byte range to read — runs your code on each in parallel, and brings the result back.

The mental shift from pandas: pandas runs your code in one process, on one machine, against memory it owns directly. Spark runs a *plan* you describe — across many processes, on many machines, against memory those processes own. You never move data between machines by hand; Spark does it for you.


## The cluster — driver, executors, cluster manager

Three roles whenever Spark runs:

- **Driver** — the process running your Python script. Owns the SparkSession, builds the execution plan, decides what work to send out. The project manager — never lifts a box itself.
- **Executors** — worker processes that hold partitions in memory and run the actual computations. The delivery drivers, each handling their own route.
- **Cluster manager** — allocates machines to your Spark application. It does not understand your job; it just hands out resources. Options include Standalone, YARN, Kubernetes, or **local mode** (driver and executors collapsed into one JVM on your laptop).

![Spark Cluster Architecture](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-cluster-architecture.svg)


## Memory-first execution

The big leap over the previous generation, Hadoop MapReduce, is one design choice: keep intermediate results in executor RAM instead of writing them to disk between every step. A multi-step pipeline used to do a full disk round-trip per step. In Spark, data stays in memory until something forces it out.

That single decision is most of Spark's speed advantage. Caching, broadcast joins, and the rest are details on top of it — covered in notebook 08.


## From your code to distributed work

When you write `df.filter(...).groupBy(...).count()`, here is what happens behind the scenes:

1. The driver builds a **logical plan** from your DataFrame chain.
2. The **Catalyst** optimizer rewrites it — pushing filters down, reordering joins, dropping unused columns.
3. The plan becomes a **physical plan**, then a **DAG** of stages.
4. Each stage splits into **tasks** — one task per partition.
5. Tasks ship to executors and run in parallel across their CPU cores.

Three exam-vocabulary words to lock in now:

- **Job** — what one action triggers. One `.count()` call → one job.
- **Stage** — a slice of work between shuffles. A new shuffle starts a new stage.
- **Task** — the smallest unit of work. One partition, processed by one executor core.

![Job, Stages, Tasks](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-job-stages-tasks.svg)


## Deployment modes

How the driver and executors actually get placed onto machines:

- **`local[*]`** — everything in one JVM on your laptop. The mode this notebook uses. The `*` means "use all available cores."
- **Standalone** — Spark's built-in cluster manager. Simple, no external dependency.
- **YARN** — runs on Hadoop clusters. Common in legacy on-prem stacks.
- **Kubernetes** — driver and executors as pods. The modern cloud-native default.
- **Databricks** — managed Spark; the cluster manager is hidden behind the platform.

For learning, `local[*]` is plenty. For the exam, you should at minimum recognize each name and what it runs on.

![Spark Setup Options](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-setup-options.svg)


## When NOT to use Spark

Before standing up a session, a word on when Spark is the wrong tool:

- **Your data fits comfortably in pandas.** Below roughly ten gigabytes on a modern laptop, pandas is faster, simpler, and has no cluster overhead.
- **You need single-row, sub-millisecond lookups.** Spark is throughput-optimized, not latency-optimized. Reach for Postgres, DynamoDB, or Redis.
- **You need OLTP semantics.** Spark is for analytical workloads — read-heavy, write-batched. It is not a transactional database.
- **You only need a scheduled script.** A `cron` plus a Python script can be the right answer; reach for Spark when scale or fault tolerance actually demand it.

The honest rule: Spark earns its complexity when single-machine tools stop working.


## Setup — your first SparkSession

Local install (Java 17+ must already be present):

```bash
pip install pyspark==3.5.3
```

The pattern below is what every notebook in the series uses: build a `SparkSession`, name the app, point at `local[*]`, call `getOrCreate()`. The returned object is your handle to everything Spark — reading data, registering views, running queries.

**Key note — `getOrCreate()` is sticky.** If a SparkSession already exists in your kernel (from a previous cell, a previous notebook, or anywhere else), `builder.getOrCreate()` returns *that* session and **silently ignores your new configs**. To pick up new config in a running notebook, call `spark.stop()` first, then build again.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkFoundations")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

spark.version


## Lazy evaluation — first taste

Spark operations split into two kinds:

- **Transformations** — `filter`, `select`, `groupBy`, `join`. They build the recipe. Nothing runs.
- **Actions** — `count`, `show`, `collect`, `write`. They trigger execution against the recipe.

Why this matters: by the time you call an action, Spark sees the *entire* chain of transformations and can plan the most efficient execution. If transformations ran eagerly, each step would commit before the next, and the optimizer would have nothing to optimize.

![Lazy Evaluation](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-lazy-evaluation.svg)


In [ ]:
# A tiny in-memory DataFrame of bank customers — enough to demonstrate Spark mechanics.
data = [
    ("CUST0001", "Aarav Sharma",  "Mumbai",    780),
    ("CUST0002", "Priya Nair",    "Delhi",     650),
    ("CUST0003", "Rohan Gupta",   "Bengaluru", 720),
    ("CUST0004", "Anjali Mehta",  "Pune",      810),
    ("CUST0005", "Vikram Reddy",  "Hyderabad", 590),
]
columns = ["customer_id", "full_name", "city", "credit_score"]
df = spark.createDataFrame(data, columns)

# Chain two transformations. Notice: no output, no work done.
high_credit = df.filter(df.credit_score >= 700).select("customer_id", "full_name", "credit_score")
print("Transformations defined. Nothing has executed yet.")


In [ ]:
# Inspect the plan Spark built up. Catalyst will already have pushed the filter close to the scan.
high_credit.explain()


In [ ]:
# .show() is an action — this is the call that actually triggers execution.
high_credit.show()


## Hello Spark — a multi-step query

The cells above already proved the install: a SparkSession started, a DataFrame was built, an action ran. One more pass to confirm a multi-step query works end to end — same five customers, this time grouped through SQL.


In [ ]:
# Register the DataFrame as a temporary SQL view, then query it.
df.createOrReplaceTempView("customers")

spark.sql("""
    SELECT city,
           COUNT(*)                    AS num_customers,
           ROUND(AVG(credit_score), 0) AS avg_credit_score
    FROM customers
    GROUP BY city
    ORDER BY avg_credit_score DESC
""").show()


# Execution Model

You've seen the rule from the outside: transformations are lazy, actions trigger work. The rest of the notebook peels back the layers — *why* lazy evaluation matters, what Spark does with the visibility it buys, and how to inspect the result.

Four layers, top to bottom:

- **Catalyst** — the rule-based optimizer that rewrites your plan in four phases before any executor sees it.
- **Tungsten** — the runtime layer that makes the optimized plan fast on the JVM (off-heap memory, whole-stage codegen).
- **`.explain()`** — your window into all of the above.
- **Adaptive Query Execution (AQE)** — runtime re-optimization that adjusts the plan *after* shuffles, using real statistics instead of estimates.

The thread tying these together: **Catalyst can only optimize what it can see, and it can only see the whole pipeline because nothing has run yet.** Lazy evaluation is the precondition; Catalyst, Tungsten, and AQE are what Spark does with that visibility.


## Lazy evaluation, in depth

The first-taste section above gave you the rule. Here is the table to keep handy when reading other people's code:

| | Transformations | Actions |
|---|---|---|
| Execution | Lazy — append to the plan | Eager — trigger DAG execution |
| Returns | Another DataFrame | A value, file, or printed output |
| Examples | `select`, `filter`, `withColumn`, `groupBy`, `join`, `orderBy` | `show`, `count`, `collect`, `take`, `first`, `write` |

Because it's lazy, Spark will:

- **Push filters early** — a filter declared at the end of the chain may execute right next to the data scan.
- **Prune projections** — columns you never reference are dropped before reading.
- **Fuse operations** — multiple narrow transformations collapse into one tight loop (whole-stage codegen, two sections down).
- **Skip everything** — if you define transformations and never call an action, no work runs.


## The four Catalyst plans

When you call an action, your DataFrame chain travels through Catalyst as four distinct plans before hitting the cluster.

1. **Parsed logical plan** — your code as a tree of unresolved references. Column names are still strings; types are unknown.
2. **Analyzed logical plan** — Catalyst resolves column names against the schema, checks types, and binds function calls. Errors here look like `AnalysisException`.
3. **Optimized logical plan** — rule-based rewrites: predicate pushdown (filters move toward the scan), projection pruning (drop unused columns), constant folding (`1 + 2 → 3`), reordering, eliminating redundant projects.
4. **Physical plan** — Catalyst picks operators (`HashAggregate` vs `SortAggregate`, `BroadcastHashJoin` vs `SortMergeJoin`) and a physical strategy. The output is what actually runs on executors.

![Catalyst Phases](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-catalyst-phases.svg)

You can inspect each plan with `.explain(extended=True)` — demonstrated below.


## Tungsten — making the JVM go fast

Catalyst gives you the right *plan*. **Tungsten** makes that plan run fast on the JVM. Two ideas:

- **Off-heap memory management** — Spark allocates structured binary buffers off the JVM heap, bypassing the garbage collector. Less GC pause, more cache-friendly memory layouts.
- **Whole-stage code generation (WSCG)** — for each stage, Catalyst generates a single tightly-fused JVM method that processes a row through every operator at once (filter → project → aggregate). One loop instead of one method call per operator per row.

This is why DataFrames beat RDDs on structured data: RDDs have no schema, so Tungsten cannot lay them out efficiently or generate per-stage code. With DataFrames, the schema is a contract Tungsten can compile against.

The proof lands in the next section: the physical plan for a DataFrame query is wrapped in a `*(N) WholeStageCodegen` block.


## `.explain()` modes — seeing inside Catalyst

`df.explain(mode="...")` exposes the plans Catalyst built. Five modes:

| Mode | Shows |
|---|---|
| `"simple"` (default) | Just the physical plan |
| `"extended"` | All four plans (parsed → analyzed → optimized → physical) |
| `"codegen"` | The generated JVM code per stage |
| `"cost"` | Optimized plan with row-count estimates per node |
| `"formatted"` | Physical plan + a separate operator-details section |

Same query, multiple lenses. Below: a small group-by aggregation against the five-customer DataFrame, viewed in three modes.


In [ ]:
from pyspark.sql.functions import avg

query = (
    df
    .filter(df.credit_score >= 600)
    .groupBy("city")
    .agg(avg("credit_score").alias("avg_score"))
)

print("=" * 70)
print("simple — physical plan only")
print("=" * 70)
query.explain()


In [ ]:
print("=" * 70)
print("extended — all four plans")
print("=" * 70)
query.explain(extended=True)


In [ ]:
print("=" * 70)
print("formatted — physical plan + operator details")
print("=" * 70)
query.explain(mode="formatted")


## Adaptive Query Execution (AQE)

Before Spark 3.0, Catalyst made all decisions upfront using *estimated* statistics. **AQE re-optimizes the plan at runtime**, after each shuffle, using *real* partition statistics.

Three things AQE actually does:

- **Dynamic coalescing** — after a shuffle produces, say, 200 partitions but most are tiny, AQE merges them into a smaller number of right-sized partitions. Fewer task-startup overheads.
- **Dynamic join strategy switch** — a sort-merge join can be promoted to a broadcast join at runtime if one side turns out small enough.
- **Skew join handling** — splits oversized partitions so one slow task doesn't block a stage. *Covered in detail in notebook 08 (Performance Tuning).*

AQE is enabled by default since Spark 3.2. Below: the same `groupBy` aggregation with AQE on and off, observing the post-shuffle partition count change.


In [ ]:
# AQE on: post-shuffle coalescing reduces the partition count.
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE on  : output partitions =", query.rdd.getNumPartitions())

# AQE off: post-shuffle stays at spark.sql.shuffle.partitions (default 200).
spark.conf.set("spark.sql.adaptive.enabled", "false")
query_no_aqe = (
    df
    .filter(df.credit_score >= 600)
    .groupBy("city")
    .agg(avg("credit_score").alias("avg_score"))
)
print("AQE off : output partitions =", query_no_aqe.rdd.getNumPartitions())

# Re-enable AQE for any later cells.
spark.conf.set("spark.sql.adaptive.enabled", "true")


# Spark Connect

Everything above assumed the classic model: your Python process *is* the driver, talking to a JVM running inside the same process. **Spark Connect breaks that link** — the driver becomes a long-lived server, and your client is a thin library that ships plans over gRPC.

The DataFrame API does not change. What changes is *where* the JVM lives, *who* owns it, and what kinds of code can run on the client.


## The split-driver architecture

In **classic mode**, your Python (or Scala) process *is* the driver. PySpark talks to a JVM in the same process via Py4J; that JVM holds `SparkContext` and coordinates the executors. Your script and the Spark driver share a lifetime — if the script exits, the driver dies.

**Spark Connect** breaks that link. The Spark driver runs as a long-lived server. Your client is a thin library that builds an unresolved logical plan, sends it over gRPC, and receives results back as Arrow batches. The cluster keeps running between client connections.

Think of it like JDBC: the database server is one process, your application is another, and they speak a protocol. Spark Connect does the same for Spark.

```text
Classic mode                              Spark Connect mode
─────────────────────────                 ────────────────────────────────────
  ┌─────────────────────┐                   ┌──────────────┐         ┌──────────────────────┐
  │  Your Python proc.  │                   │  Client      │  gRPC   │  Spark Connect       │
  │  ┌───────────────┐  │                   │  (Python,    │ ──────► │  Server (JVM)        │
  │  │  Py4J ↔ JVM   │  │                   │   Scala, R,  │ ◄────── │  ┌─────────────────┐ │
  │  │  Driver       │  │                   │   Go, ...)   │ Arrow   │  │  Driver         │ │
  │  └───────┬───────┘  │                   └──────────────┘ batches │  └────────┬────────┘ │
  │          │          │                                            │           │          │
  │          ▼          │                                            │           ▼          │
  │      Executors      │                                            │       Executors      │
  └─────────────────────┘                                            └──────────────────────┘
  one process = one app                                              one server = many clients
```

The client side has **no JVM**. That's the whole point — your client can be a tiny Python script, a Jupyter notebook, an IDE, or a Go program. None of them need to bundle Spark.


## Why Spark Connect exists

The classic model has friction in four places that Connect fixes:

| Problem with classic | What Connect changes |
|---|---|
| Client and driver share one JVM — version mismatches, dependency conflicts, classpath surprises | Client is a thin library; server controls all JVM dependencies |
| Hard to use Spark from non-JVM languages — every language needs a Py4J-like bridge | gRPC protocol — any language with a gRPC stub can be a client |
| Long-lived development sessions are awkward (notebook restart kills the driver) | Server lives independently; clients connect and disconnect freely |
| Hard to upgrade Spark — every client must match the cluster | Client and server can be on different Spark versions (within protocol compatibility) |

Spark Connect became Generally Available in **Spark 3.5**. **Databricks Connect** (the proprietary client used by IDEs against Databricks workspaces) is built on top of it.


## Starting a server and connecting from a client

On a machine with Spark installed:

```bash
# Start the Connect server (defaults to port 15002)
./sbin/start-connect-server.sh \
    --packages org.apache.spark:spark-connect_2.12:3.5.3

# Stop it
./sbin/stop-connect-server.sh
```

From the **client** side, connect with `SparkSession.builder.remote(...)`:

```python
from pyspark.sql import SparkSession

# Connect to a remote Spark Connect server
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

# Then use the DataFrame API exactly as before
df = spark.read.parquet("s3://bucket/data/")
df.filter("amount > 100").groupBy("category").count().show()
```

The URL scheme is `sc://host:port`. The same can be set via the `SPARK_REMOTE` environment variable, which `getOrCreate()` picks up automatically:

```bash
export SPARK_REMOTE="sc://localhost:15002"
python my_script.py
```

Authentication, TLS, and session tokens are passed as URL parameters:

```python
spark = SparkSession.builder.remote(
    "sc://myserver.example.com:443/;token=abc123;use_ssl=true"
).getOrCreate()
```


## Detecting which mode you're in

Builders look identical at the call site but resolve to **two different `SparkSession` classes**:

- `pyspark.sql.session.SparkSession` — classic
- `pyspark.sql.connect.session.SparkSession` — Connect

Same method names, different capabilities. **This is a real exam trap** — and you can also walk into it accidentally if `SPARK_REMOTE` is set in your environment, because `getOrCreate()` will silently hand you a Connect session even when you wrote a `builder` chain that looks classic.

To check which one you have, inspect the module of the SparkSession class. If `"connect"` appears in the module name, you're in Connect mode.


In [ ]:
import os

module = type(spark).__module__
is_connect = "connect" in module

print(f"SparkSession class : {type(spark).__name__}")
print(f"Module             : {module}")
print(f"Connect mode?      : {is_connect}")

# Env vars that flip you into Connect mode without asking:
for var in ("SPARK_REMOTE", "SPARK_CONNECT_MODE_ENABLED"):
    print(f"  {var:30s} = {os.environ.get(var, '(not set)')}")


## What works and what doesn't in Connect mode

The DataFrame and SQL APIs are the supported surface — that's deliberate, because those describe a *plan* that can be serialized. Anything that depends on direct JVM access from the client cannot work.

| API | Connect | Notes |
|---|---|---|
| `DataFrame` / `Dataset` operations | ✅ | The whole reason Connect exists |
| `spark.sql(...)` | ✅ | SQL is serializable |
| `spark.read.*` / `df.write.*` | ✅ | Paths are resolved server-side |
| Built-in functions (`pyspark.sql.functions`) | ✅ | |
| Pandas UDFs, regular UDFs | ✅ | Code is shipped to the server and executed there |
| Structured Streaming | ✅ | Supported in 3.5+ |
| Pandas API on Spark (`pyspark.pandas`) | ✅ | Supported in 3.5+ |
| `spark.sparkContext` | ❌ | No `SparkContext` on the client — there's no JVM here |
| RDD API (`rdd.map`, `parallelize`, ...) | ❌ | RDDs are not part of the Connect protocol |
| `spark._jvm`, `spark._jsc` | ❌ | Internal Py4J handles do not exist |
| Adding JARs from the client (`--jars`, runtime injection) | ❌ | JARs must be on the server's classpath |
| `spark.catalog` listings | ✅ (most) | Some metastore operations are restricted |

**The takeaway for the exam:** if a question mentions `SparkContext`, `RDD`, or injecting JARs from the client — that code is incompatible with Spark Connect. If it sticks to DataFrame, SQL, and built-in or registered UDFs, it works.


In [ ]:
spark.stop()
